In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
from plot_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:


fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

df_metadata  =transform_df(df_metadata)
df_metadata

In [ ]:


def get_sel_stuff(species,inoculumn, polarize=True, sel_period=(1,3)):
    fname1 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{species}/{inoculumn}_parent1_info.csv'
    fname2 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{species}/{inoculumn}_parent2_info.csv'
    sel_stuff1 = pd.read_csv(fname1).rename(columns={'mesocosms':'mesocosm'})
    sel_stuff2 = pd.read_csv(fname2).rename(columns={'mesocosms':'mesocosm'})
    
    fname1=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{species}/{inoculumn}_parent1_info.csv'
    fname2=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{species}/{inoculumn}_parent1_info.csv'
    freq1 = pd.read_csv(fname1)
    freq2=pd.read_csv(fname2)
    df_both = get_both_dfs(fname1)
    e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
    df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
    df_both=df_both.loc[df_both['total_shift']<.1,:]

    df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
    sel_stuff1=sel_stuff1.loc[sel_stuff1['sample1'].isin(df_both_meta['sample'].unique())*sel_stuff1['sample2'].isin(df_both_meta['sample'].unique()),:]
    sel_stuff1=sel_stuff1.loc[(sel_stuff1['passage1']==sel_period[0])*(sel_stuff1['passage2']==sel_period[1]),:]
    sel_stuff2=sel_stuff2.loc[sel_stuff2['sample1'].isin(df_both_meta['sample'].unique())*sel_stuff2['sample2'].isin(df_both_meta['sample'].unique()),:]
    sel_stuff2=sel_stuff2.loc[(sel_stuff2['passage1']==sel_period[0])*(sel_stuff2['passage2']==sel_period[1]),:]
    
    df_both_meta_pfinal = df_both_meta.loc[df_both_meta['passage']== 7,:].set_index('mesocosm')
    df_both_meta_p_pol = df_both_meta.loc[df_both_meta['passage']== 0,:].set_index('mesocosm')
    df_both_meta_p_polgr=df_both_meta_p_pol.groupby(['type_mesocosm']).median(numeric_only=True).reset_index()
    to_repol = df_both_meta_p_polgr['actual_med1']>.5
    if len(to_repol) <1:
        return pd.DataFrame()
#    print(to_repol)
    to_repol=to_repol[0]
  #  print(to_repol)
    
    sel_stuff1=sel_stuff1.set_index('mesocosm')
    sel_stuff2=sel_stuff2.set_index('mesocosm')

    good_mesos = np.intersect1d(sel_stuff1.index.values,df_both_meta_pfinal.index.values)
    sel_stuff1=sel_stuff1.loc[good_mesos,:]
    sel_stuff2=sel_stuff2.loc[good_mesos,:]
    df_both_meta_pfinal=df_both_meta_pfinal.loc[good_mesos,:]

    df_both_meta_pfinal['plot_freq_act']=df_both_meta_pfinal['actual_med1']
    if to_repol:
        df_both_meta_pfinal['plot_freq_act']=df_both_meta_pfinal['actual_med2']
    df_both_meta_pfinal=df_both_meta_pfinal.loc[sel_stuff1.index.values,:]
    
    df_both_meta_pinitial=df_both_meta.loc[df_both_meta['passage']== sel_period[1],:].set_index('mesocosm')
    df_both_meta_piniall_dfs_p7tial=df_both_meta_pinitial.loc[good_mesos,:]

    df_both_meta_pfinal['sel_coeff'] = np.nan 
    df_both_meta_pfinal=df_both_meta_pfinal.loc[sel_stuff1.index.values,:]
    df_both_meta_pfinal['sel_coeff']= sel_stuff1['sel_med']
    df_both_meta_pfinal['plot_freq_initial'] = df_both_meta_pinitial['actual_med1']
    if to_repol:
        df_both_meta_pfinal=df_both_meta_pfinal.loc[sel_stuff2.index.values,:]
        df_both_meta_pfinal['sel_coeff']= sel_stuff2['sel_med']
        df_both_meta_pfinal['plot_freq_initial'] = df_both_meta_pinitial['actual_med2']


    df_both_meta_pfinal['dt'] = 7-sel_period[1]
    df_both_meta_pfinal['plot_pred']= np.exp(df_both_meta_pfinal['dt']*df_both_meta_pfinal['sel_coeff'])
    df_both_meta_pfinal['plot_pred']=df_both_meta_pfinal['plot_pred']*df_both_meta_pfinal['plot_freq_initial']/(\
            1-df_both_meta_pfinal['plot_freq_initial']+df_both_meta_pfinal['plot_freq_initial']*df_both_meta_pfinal['plot_pred'])

    
    return df_both_meta_pfinal


In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'):
            df=get_sel_stuff(sp,ino, sel_period=(0,7)).reset_index()
       # except:
        #    continue
            df['species_id']=sp
            if len(df)>0:
                all_dfs.append(df)
all_dfs=pd.concat(all_dfs)
all_dfs['species-type_meso']=all_dfs['species_id'].astype(str)+'-'+all_dfs['type_mesocosm']
all_dfs_gr=all_dfs.groupby(['species-type_meso']).median(numeric_only=True).reset_index()
bad_inos = ['100146-AA-AF-mBHI',
 '100146-AE-AF-mBHI',
 '102478-AA-AF-mBHI',
 '102478-AE-AF-mBHI',
 '100196-AA-AF-mBHI',
 '100196-AE-AF-mBHI',
 '100196-AA-AE-mGAM',
 '100196-AE-AF-mGAM',
 '102544-AA-AF-mBHI',
 '102544-AE-AF-mBHI',
 '101349-AA-AF-mBHI',
 '101349-AE-AF-mBHI',
 '102506-AA-AE-mBHI',
 '102506-AE-AF-mBHI',
 '102506-AA-AF-mBHI',
 '102506-AE-AF-mBHI',
 '102506-AA-AE-mGAM',
 '102506-AE-AF-mGAM',
 '101346-AA-AE-mBHI',
 '101346-AE-AF-mBHI',
 '101346-AA-AF-mBHI',
 '101346-AE-AF-mBHI',
 '101346-AA-AE-mGAM',
 '101346-AE-AF-mGAM',
 '101346-AA-AE-mBHI',
 '101346-AA-AF-mBHI',
 '100120-AA-AF-mBHI',
 '100120-AE-AF-mBHI',
 '102327-AA-AF-mBHI',
 '102327-AE-AF-mBHI']
all_dfs['species-ino']=all_dfs['species_id'].astype(str) +'-'+ all_dfs['inoculumn'].astype(str)
all_dfs = all_dfs.loc[~all_dfs['species-ino'].isin(bad_inos),:]
all_dfs

In [ ]:
all_dfs_gr['sel_coeff_abs']=all_dfs_gr['sel_coeff'].abs()
good_sel = all_dfs_gr['sel_coeff']
good_sel = good_sel[~np.isnan(good_sel)]
frequencies, edges = np.histogram(good_sel,bins=20)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=500, height=200, xlabel='Sel Coefficient Passages 0-7', 
                                        ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )

In [ ]:

def get_both_dfs(fname):
    #print(fname)
    df1 = pd.read_csv(fname).drop(columns='Unnamed: 0').set_index('sample')
    df2 = pd.read_csv(fname.split('parent1_info.csv')[0] + 'parent2_info.csv').drop(columns='Unnamed: 0').set_index('sample')
    df2['conf_int'] = df2['boot_high']-df2['boot_low']
    
    df1['conf_int'] = df1['boot_high']-df1['boot_low']
    df_both = pd.concat([df1.rename(columns = {'boot_med':'boot_med1', 'boot_low':'boot_low1', 'boot_high':'boot_high1',
       'actual_med':'actual_med1', 'conf_int':'conf_int1'}),
                         df2.rename(columns = {'boot_med':'boot_med2', 'boot_low':'boot_low2', 'boot_high':'boot_high2',
       'actual_med':'actual_med2', 'conf_int':'conf_int2'})],axis=1)
    
    df_both['species'] = fname.split('/')[-2]
    df_both['fname'] = fname
   # print('-'.join(fname1.split('/')[-1].split('-')[:-1]))
   # parent_media = 
    
    df_both['subjects_measured'] = '-'.join(fname.split('/')[-1].split('-')[:-1])
    df_both['in_measured'] = '-'.join(fname.split('/')[-1].split('_')[:-2])
    df_both['total_shift'] = np.abs(1-(df_both['actual_med1'] + df_both['actual_med2']))
    df_both['total_shift12'] = np.abs(1-(df_both['boot_low1'] + df_both['boot_high2']))
    df_both['total_shift21'] = np.abs(1-(df_both['boot_low2'] + df_both['boot_high1']))
    df_both['total_shift_max'] = df_both['total_shift21']
    df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift_max'] = df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift12']

    return df_both

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4.csv').set_index('sample')

for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        fname1=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if os.path.exists(fname1):
            
            df_both=get_both_dfs(fname1)
            df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
            df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
       # except:
        #    continue
            df_both_meta['species_id']=sp
            df_both_meta['diff_from_in1']=df_both_meta['actual_med1']
            df_both_meta['diff_from_in2']=df_both_meta['actual_med2']
            in_samples = df_both_meta['inoculumn_sample'].unique()
            if len(in_samples) != len(df_both_meta.loc[df_both_meta['sample'].isin(in_samples),'actual_med1']):
                continue
            
            for in_sample in df_both_meta['inoculumn_sample'].unique():
                
                with_in_sample = df_both_meta.loc[df_both_meta['inoculumn_sample']==in_sample,'sample'].values
                in1_val=df_both_meta.loc[df_both_meta['sample']==in_sample,'actual_med1'].values[0]
                df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in1']= \
                    df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in1'] - in1_val
                
                in2_val=df_both_meta.loc[df_both_meta['sample']==in_sample,'actual_med2'].values[0]
                df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in2']= \
                    df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in2'] - in2_val
            all_dfs.append(df_both_meta)
all_dfs=pd.concat(all_dfs)
all_dfs['species-type_meso']=all_dfs['species_id'].astype(str)+'-'+all_dfs['type_mesocosm']
all_dfs['species-ino']=all_dfs['species_id'].astype(str) +'-'+ all_dfs['inoculumn'].astype(str)
all_dfs = all_dfs.loc[~all_dfs['species-ino'].isin(bad_inos),:]
all_dfs

In [ ]:
all_dfs['species-mesocosm']=all_dfs['species_id'].astype(str) + '-' + all_dfs['mesocosm'].astype(str)
all_dfs7 = all_dfs.loc[all_dfs['passage']==7,:]
all_dfs0 = all_dfs.loc[all_dfs['passage']==0,:]
all_dfs7['passage_0'] = np.nan
all_dfs7['sp_ino']=all_dfs7['species_id'].astype(str)+'-'+all_dfs7['inoculumn_sample']
all_dfs0['sp_ino']=all_dfs0['species_id'].astype(str)+'-'+all_dfs0['inoculumn_sample']
for spin in all_dfs7['sp_ino'].unique():
    in_sample = '-'.join(spin.split('-')[1:])
    all_dfs0_good=all_dfs0.loc[all_dfs0['sp_ino']==spin,:]
    all_dfs0_good=all_dfs0_good.loc[all_dfs0_good['sample']==in_sample,'actual_med1'].values
    if len(all_dfs0_good)==0:
        continue
    all_dfs7.loc[all_dfs7['sp_ino']==spin,'passage_0']=all_dfs0_good[0]

    

In [ ]:
all_dfs7 = all_dfs7.loc[~all_dfs7['passage_0'].isna(),:]
all_dfs7p = all_dfs7.copy()
all_dfs7p.loc[all_dfs7p['passage_0']>.5,'actual_med1']=1.-all_dfs7p.loc[all_dfs7p['passage_0']>.5,'actual_med1']
all_dfs7p.loc[all_dfs7p['passage_0']>.5,'passage_0']=1.-all_dfs7p.loc[all_dfs7p['passage_0']>.5,'passage_0']
all_dfs7p.head()

In [ ]:

all_dfs7p['passage'] = all_dfs7p['passage'].astype(str)

p1 = hv.Points(all_dfs7p,vdims = ['passage_0','actual_med1'],
              kdims = [hv.Dimension('passage',values = ['0','7']),'actual_med1']).opts(jitter = .1, size=4,
                                                      color = 'passage_0',
                                                      show_legend=False,
                                                      xlabel='Passage',cmap = 'Magma',
                                                                                       
                                                      
                                                      #cmap = bokeh.palettes.
                                                      alpha = .5)
all_dfs7p0=all_dfs7p.copy()
all_dfs7p0['passage']='0'
all_dfs7p0['actual_med1']=all_dfs7p0['passage_0'].copy()
p2= hv.Points(all_dfs7p0,vdims = ['passage_0','actual_med1'],
              kdims = [hv.Dimension('passage',values = ['0','7']),'actual_med1']).opts(jitter = .1, size=4,
                                                      color = 'passage_0',
                                                      show_legend=False,
                                                      xlabel='Passage',ylabel = 'Strain Frequency',
                                                                                       
                                                
                                                      cmap = 'Magma',
                                                      alpha = .5)
p = hv.Scatter(all_dfs7p, vdims = ['inoculumn','passage_0'],
               kdims = ['passage_0','actual_med1'],).opts(color='inoculumn', size=6,
                                                          xlabel='Freq Passage 0',
                                                          ylabel = 'Freq Passage 7',
                                                          width = 600,alpha=.5,
                                                          cmap = bokeh.palettes.Bright[6],
                                                          line_color='black',legend_position='right',)

from scipy.stats import pearsonr
print(pearsonr(all_dfs7p['passage_0'].values,all_dfs7p['actual_med1'].values))
good_ones = all_dfs7p.loc[all_dfs7p['passage_0']>.3,:]
print(pearsonr(good_ones['passage_0'].values,good_ones['actual_med1'].values))
good_ones = all_dfs7p.loc[all_dfs7p['passage_0']<.3,:]
print(pearsonr(good_ones['passage_0'].values,good_ones['actual_med1'].values))
p=hv.render(p)
p.xaxis.axis_label_text_font_style = 'normal'
p.yaxis.axis_label_text_font_style = 'normal'
bokeh.io.show(p)
p.output_backend='svg'
export_plot_pdf(p,'07corr')

In [ ]:
for ino in all_dfs7p['inoculumn'].unique():
    good_ones = all_dfs7p.loc[all_dfs7p['inoculumn']==ino,]
    print(ino, pearsonr(good_ones['passage_0'].values,good_ones['actual_med1'].values))
    if len(good_ones.loc[good_ones['passage_0']<.1, 'passage_0'].values)>0:
        print(ino,  'small',pearsonr(good_ones.loc[good_ones['passage_0']<.1, 'passage_0'].values, 
                             good_ones.loc[good_ones['passage_0']<.1, 'actual_med1'].values))
    if len(good_ones.loc[good_ones['passage_0']>.1, 'passage_0'].values):
        print(ino, 'big',  pearsonr(good_ones.loc[good_ones['passage_0']>.1, 'passage_0'].values,
                                    good_ones.loc[good_ones['passage_0']>.1, 'actual_med1'].values))

In [ ]:
all_dfs=all_dfs.loc[all_dfs['total_shift']<.1,:]


In [ ]:
all_dfs=all_dfs.loc[all_dfs['total_shift']<.1,:]
all_dfs_p7 = all_dfs.loc[all_dfs['passage']==7,:]
all_dfs_p7['diff_from_in1_abs']=all_dfs_p7['diff_from_in1'].abs()
all_dfs_gr=all_dfs_p7.groupby(['species-type_meso']).median(numeric_only=True).reset_index()
all_dfs_gr['diff_from_in1_abs']=all_dfs_gr['diff_from_in1'].abs()
frequencies, edges = np.histogram(all_dfs_p7['diff_from_in1_abs'],bins=20)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=400, height=200, xlabel='|Absolute Frequency Shift|', 
                                        ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )

In [ ]:
all_dfs_p7['dir']=all_dfs_p7['diff_from_in1']>0
all_dfs_p7['counts']=1
all_dfs_p7_gr=all_dfs_p7.groupby(['species-type_meso','species_id']).sum(numeric_only=True)
all_dfs_p7_gr['frac_one']=all_dfs_p7_gr['dir']/all_dfs_p7_gr['counts']
all_dfs_p7_gr['dir_adjust'] = all_dfs_p7_gr['frac_one']<.5
all_dfs_p7_gr['counts_major']=all_dfs_p7_gr['dir']
all_dfs_p7_gr['counts_opp'] = all_dfs_p7_gr['counts']-all_dfs_p7_gr['dir']
all_dfs_p7_gr.loc[all_dfs_p7_gr['dir_adjust'],'counts_major']=all_dfs_p7_gr['counts_opp'].astype(int)

all_dfs_p7_gr['all_one_dir']=all_dfs_p7_gr['counts_major']==all_dfs_p7_gr['counts']
all_dfs_p7_gr['counts_major'].min()
all_dfs_p7_gr=all_dfs_p7_gr.loc[all_dfs_p7_gr['counts']>1,:]
len(all_dfs_p7_gr.reset_index()['species_id'].unique())

In [ ]:
counts_34 = all_dfs_p7_gr.loc[all_dfs_p7_gr['counts']>2,:]
cons = len(counts_34.loc[counts_34['all_one_dir'],:])
trials_4 = len(counts_34.loc[counts_34['counts']==4,:])
trials_3 = len(counts_34.loc[counts_34['counts']==3,:])

In [ ]:
def get_bs_reps_all_4s(n_trials,n_bs_trials=10000):
    bs_reps = np.zeros(n_bs_trials)
    bs_reps1=  np.zeros(n_bs_trials)
    bs_reps2 = np.zeros(n_bs_trials)
    for n in range(n_bs_trials):
        rep = np.random.binomial(4,1/2,size=n_trials)
        bs_reps[n]=np.sum(rep==4)+np.sum(rep==0)
        bs_reps1[n]=np.sum(rep==3)+np.sum(rep==1)
        bs_reps2[n]=np.sum(rep==2)
    return bs_reps, bs_reps1,bs_reps2

def get_bs_reps_4s_3s(n_trials4, n_trials3, n_bs_trials=10000):
    bs_reps = np.zeros(n_bs_trials)
    for n in range(n_bs_trials):
        rep4 = np.random.binomial(4,1/2,size=n_trials4)
        rep3=np.random.binomial(3,1/2,size=n_trials3)
        bs_reps[n]=np.sum(rep4==4)+np.sum(rep4==0)+np.sum(rep3==3)+np.sum(rep3==0)
    return bs_reps
        

In [ ]:
b, b1,b2=get_bs_reps_all_4s(46,n_bs_trials=100000)
b = b
b34=get_bs_reps_4s_3s(trials_4, trials_3, n_bs_trials=100000)/(trials_4+trials_3)
b34

In [ ]:
trials_4+trials_3

In [ ]:
dfgood = pd.DataFrame(data = {'Data': ['obs','null'], 'Vals': [cons/(trials_4+trials_3), np.median(b34)]})
p = hv.Bars(dfgood.sort_values(by='Data',ascending=False), kdims = ['Data'],vdims = ['Vals']).opts(ylim=(0,1),fill_color='Data',
                                                                                  cmap = [bokeh.palettes.Vibrant[4][-1], 'grey',],
                                                                                   alpha = .5,
                                                                                   bar_width = .5,show_legend=False)

low = np.median(b34)-np.percentile(b34, 2.5)
upp = np.percentile(b34, 97.5) - np.median(b34)
print(upp)
print(low)
print(np.median(b34))
p=hv.render(p)
from bokeh.models import ColumnDataSource, Whisker
source = ColumnDataSource(data=dict(base=['null'], upper=[np.percentile(b34, 97.5)], lower=[np.percentile(b34, 2.5)]))
error = Whisker(base="base", upper="upper", lower="lower", source=source,
                level="annotation", line_width=2)

p.add_layout(error)
p.output_backend='svg'
bokeh.io.show(p)
export_plot_pdf(p,'sim_vs_null_repro')